In [ ]:
!pip install ragas langchain-openai openpyxl pandas datasets langchain-anthropic -q


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 466.5/466.5 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.5/88.5 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 627.7/627.7 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.2/178.2 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the

In [ ]:
import ragas
import langchain_openai
import openpyxl
import pandas
import datasets
print("All imports OK!")

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


All imports OK!


In [ ]:
from google.colab import userdata
import os

os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")

In [ ]:
import json, os, warnings
warnings.filterwarnings("ignore")

import pandas as pd
from datasets import Dataset
from openpyxl import load_workbook
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.utils import get_column_letter
from ragas import evaluate
from ragas.metrics import faithfulness, answer_correctness

def detect_columns(df):
    col_map = {}
    for col in df.columns:
        cl = col.strip().lower()
        if "question" in cl and "question" not in col_map:
            col_map["question"] = col
        elif ("system generated" in cl or "generated answer" in cl) and "answer" not in col_map:
            col_map["answer"] = col
        elif ("source of truth" in cl or "reference answer" in cl) and "ground_truth" not in col_map:
            col_map["ground_truth"] = col
        elif "chunk 1" in cl or "retrieve chunk 1" in cl:
            col_map["chunk1"] = col
        elif "chunk 2" in cl or "retrieve chunk 2" in cl:
            col_map["chunk2"] = col
        elif "chunk 3" in cl or "retrieve chunk 3" in cl:
            col_map["chunk3"] = col
    return col_map

def score_fill(val):
    if val >= 0.85: return PatternFill("solid", start_color="C6EFCE", end_color="C6EFCE")
    if val >= 0.65: return PatternFill("solid", start_color="FFEB9C", end_color="FFEB9C")
    return PatternFill("solid", start_color="FFC7CE", end_color="FFC7CE")

def score_font(val):
    if val >= 0.85: return Font(color="276221", bold=True)
    if val >= 0.65: return Font(color="9C5700", bold=True)
    return Font(color="9C0006", bold=True)

def build_llm(provider="anthropic"):
    if provider == "anthropic":
        from langchain_anthropic import ChatAnthropic
        from ragas.llms import LangchainLLMWrapper
        llm = LangchainLLMWrapper(ChatAnthropic(
            model="claude-3-5-sonnet-20241022",
            anthropic_api_key=os.environ["ANTHROPIC_API_KEY"],
            max_tokens=2048,
        ))
        return llm, None
    else:
        from langchain_openai import ChatOpenAI, OpenAIEmbeddings
        from ragas.llms import LangchainLLMWrapper
        from ragas.embeddings import LangchainEmbeddingsWrapper
        llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4o-mini", openai_api_key=os.environ["OPENAI_API_KEY"], max_tokens=2048))
        embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(openai_api_key=os.environ["OPENAI_API_KEY"]))
        return llm, embeddings

def run_evaluation(input_file, provider="anthropic", chunk_limit=1000):
    METRICS = {
        "faithfulness":       (faithfulness,       ["question", "answer", "contexts"]),
        "answer_correctness": (answer_correctness, ["question", "answer", "ground_truth"]),
    }

    df = pd.read_excel(input_file)
    col_map = detect_columns(df)
    print("Detected columns:", col_map)

    llm, embeddings = build_llm(provider)

    def get_contexts(row):
        chunks = []
        for key in ["chunk1", "chunk2", "chunk3"]:
            if key in col_map:
                val = str(row.get(col_map[key], "")).strip()
                if val and val.lower() != "nan":
                    chunks.append(val[:chunk_limit])
        return chunks if chunks else [""]

    base_data = {
        "question":     [str(r[col_map["question"]]).strip() for _, r in df.iterrows()],
        "answer":       [str(r[col_map["answer"]]).strip() for _, r in df.iterrows()],
        "contexts":     [get_contexts(r) for _, r in df.iterrows()],
        "ground_truth": [str(r[col_map["ground_truth"]]).strip() for _, r in df.iterrows()],
    }

    score_columns = []
    for metric_name, (metric_obj, required_cols) in METRICS.items():
        print(f"\nRunning: {metric_name} ...")
        metric_obj.llm = llm
        if embeddings and hasattr(metric_obj, "embeddings"):
            metric_obj.embeddings = embeddings
        dataset = Dataset.from_dict({k: base_data[k] for k in required_cols})
        try:
            result = evaluate(dataset, metrics=[metric_obj])
            scores = result.to_pandas()[metric_name].tolist()
            col_label = metric_name.replace("_", " ").title()
            df[col_label] = [round(float(s), 3) if s is not None else None for s in scores]
            score_columns.append(col_label)
            valid = [s for s in scores if s is not None]
            print(f"  Average: {round(sum(valid)/len(valid), 3)}")
        except Exception as e:
            print(f"  ERROR: {e}")

    output_path = input_file.replace(".xlsx", "_ragas_evaluated.xlsx")
    df.to_excel(output_path, index=False)

    wb = load_workbook(output_path)
    ws = wb.active
    hfill = PatternFill("solid", start_color="1F3864", end_color="1F3864")
    hfont = Font(color="FFFFFF", bold=True, size=11)
    for cell in ws[1]:
        cell.fill = hfill
        cell.font = hfont
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
    col_names = [cell.value for cell in ws[1]]
    for row_idx in range(2, ws.max_row + 1):
        for col_name in score_columns:
            if col_name not in col_names: continue
            cell = ws.cell(row=row_idx, column=col_names.index(col_name) + 1)
            try:
                val = float(cell.value)
                cell.fill = score_fill(val)
                cell.font = score_font(val)
                cell.alignment = Alignment(horizontal="center", vertical="center")
            except: pass
    for col_idx in range(1, ws.max_column + 1):
        ws.column_dimensions[get_column_letter(col_idx)].width = 28
    wb.save(output_path)
    print(f"\nDone! Saved to: {output_path}")
    return output_path

print("Functions loaded successfully!")

Functions loaded successfully!


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving Faithfulness_Correctedness_evaluation_main.xlsx to Faithfulness_Correctedness_evaluation_main.xlsx


In [ ]:
import os
os.listdir()

['.config',
 'Faithfulness_evaluation_main.xlsx',
 'Faithfulness_evaluation_main copy.xlsx',
 'Faithfulness_evaluation_main_ragas_evaluated.xlsx',
 'Faithfulness_Correctedness_evaluation_main.xlsx',
 'sample_data']

In [ ]:
import pandas as pd

for filename in [
    "Faithfulness_Correctedness_evaluation_main.xlsx",

]:
    df = pd.read_excel(filename)
    print(f"\n{filename} — {len(df)} rows")
    print(df['Questions '].tolist())


Faithfulness_evaluation_main copy.xlsx — 15 rows
["What is 'personal data' under GDPR?", 'What is the right to restriction of processing under GDPR?', 'What obligations does GDPR impose on joint controllers?', 'What is the right to erasure under GDPR?', 'What does GDPR say about processing health data?', 'What does HIPAA say about the privacy of patient health information? ', 'What does HIPAA require after a breach of unsecured PHI?', 'What uses of PHI are permitted without authorization under HIPAA?', "What is HIPAA's treatment of psychotherapy notes, and why are they specially protected?", 'What is the workforce training requirement under HIPAA?', 'What is the NIST Cybersecurity Framework (CSF)?', 'What does NIST recommend for vulnerability management? ', 'What are NIST CSF profiles?', 'What are NIST CSF Implementation Tiers?', 'What does NIST say about security awareness and training? ']


In [ ]:
output_file = run_evaluation(
    input_file="Faithfulness_Correctedness_evaluation_main.xlsx",
    provider="ANTHROPIC_API_KEY",
    chunk_limit=6000
)

Detected columns: {'question': 'Questions ', 'ground_truth': 'Source Of Truth Answer/ Reference Answer ', 'answer': 'System Generated Answer', 'chunk1': 'Retrieve Chunk 1', 'chunk2': 'Retrieve Chunk 2', 'chunk3': 'Retrieve Chunk 3'}

Running: faithfulness ...


Evaluating:   0%|          | 0/15 [00:00<?, ?it/s]

  Average: 0.905

Running: answer_correctness ...


Evaluating:   0%|          | 0/15 [00:00<?, ?it/s]

  Average: 0.773

Done! Saved to: Faithfulness_Correctedness_evaluation_main_ragas_evaluated.xlsx


In [ ]:
from google.colab import files
files.download(output_file)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>